In [1]:
import pandas as pd
import numpy as np
import ast
import textwrap
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from shared import load_data, evaluate_model

train_df, test_df, restaurants_df = load_data()

C:\Users\nicho\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BOOLEAN_ATTRIBUTES = {
    'RestaurantsTakeOut': 'TakeOut',
    'OutdoorSeating': 'Outdoor',
    'RestaurantsDelivery': 'Deliv',
    'GoodForKids': 'GFK'
}

CATEGORICAL_ATTRIBUTES = {
    'RestaurantsPriceRange2': 'Price',
    'Ambience': 'Amb'
}

ALL_ATTRIBUTES = {**BOOLEAN_ATTRIBUTES, **CATEGORICAL_ATTRIBUTES}

def parse_attributes(attr_str):
    if pd.isna(attr_str): return {}
    return ast.literal_eval(attr_str)

def get_attr(row, attr_name):
    attrs = row.get('parsed_attributes', {})
    if not isinstance(attrs, dict): return 'Unknown'
    return str(attrs.get(attr_name, 'Unknown')).replace("u'", "").replace("'", "")

restaurants_df['parsed_attributes'] = restaurants_df['attributes'].apply(parse_attributes)

for attributes, clean_name in ALL_ATTRIBUTES.items():
    restaurants_df[clean_name] = restaurants_df.apply(lambda row: get_attr(row, attributes), axis=1)

#bin encoding
binary_clean_names = list(BOOLEAN_ATTRIBUTES.values())
binary_feature_cols = []
for col in binary_clean_names:
    bin_name = col + '_Bin'
    restaurants_df[bin_name] = restaurants_df[col].map({'True': 1, 'False': 0, 'Unknown': 0, 'None': 0}).fillna(0).astype(int)
    binary_feature_cols.append(bin_name)

#one hot encoding for attributes
categorical_clean_names = list(CATEGORICAL_ATTRIBUTES.values())

if categorical_clean_names:
    attr_dummies = pd.get_dummies(restaurants_df[categorical_clean_names], dtype=int)
else:
    attr_dummies = pd.DataFrame(index=restaurants_df.index)

#multi hot encoding for categories
restaurants_df['cat_list'] = restaurants_df['categories'].astype(str).apply(lambda x: [c.strip().lower() for c in x.split(',')])
categories_dummies = pd.get_dummies(restaurants_df['cat_list'].explode(), dtype=int).groupby(level=0).sum()

#weights
weight_binary = 0.30
weight_attr = 0.50
weight_cat = 0.20

#concat/normalize
feature_df = pd.concat([
    (restaurants_df[binary_feature_cols] * weight_binary),
    (attr_dummies * weight_attr),
    (categories_dummies * weight_cat)
], axis=1)

feature_matrix_normalized = normalize(feature_df.values, norm='l2', axis=1)
item_indices = pd.Series(restaurants_df.index, index=restaurants_df['business_id']).to_dict()

In [3]:
K_REVIEWS = 15
MIN_WORDS = 5

print(f"Selecting Top {K_REVIEWS} most recent quality reviews per restaurant...")

quality_reviews = train_df.dropna(subset=['text']).copy()
quality_reviews['datetime'] = pd.to_datetime(quality_reviews['datetime'])
quality_reviews['word_count'] = quality_reviews['text'].str.split().str.len()
quality_reviews = quality_reviews[quality_reviews['word_count'] >= MIN_WORDS]

quality_reviews = quality_reviews.sort_values(['business_id', 'datetime'], ascending=[True, False])

top_k_grouped = quality_reviews.groupby('business_id').head(K_REVIEWS)

reviews_final = top_k_grouped.groupby('business_id')['text'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
reviews_final.rename(columns={'text': 'concat_review'}, inplace=True)

restaurants_df = restaurants_df.merge(reviews_final, on='business_id', how='left')
restaurants_df['concat_review'] = restaurants_df['concat_review'].fillna("")

texts_to_encode = restaurants_df['concat_review'].apply(lambda x: ' '.join(x.split()[:380])).tolist()
model = SentenceTransformer('all-mpnet-base-v2')
text_embeddings = model.encode(texts_to_encode, show_progress_bar=True)

text_matrix_normalized = normalize(text_embeddings, norm='l2', axis=1) #L2 NORMALIZING!

print(f"Matrix shape: {text_matrix_normalized.shape}")

Selecting Top 15 most recent quality reviews per restaurant...


Batches: 100%|██████████| 24/24 [03:43<00:00,  9.31s/it]

Matrix shape: (767, 768)


In [4]:
all_train = train_df[train_df['stars'] >= 0.0]

user_meta_profiles = {}
user_text_profiles = {}

for user, group in all_train.groupby('user_id'):
    liked_item_ids = group['business_id'].tolist()
    liked_indices = [item_indices[biz] for biz in liked_item_ids if biz in item_indices]

    if liked_indices:
        user_meta_profiles[user] = np.asarray(feature_matrix_normalized[liked_indices].mean(axis=0))
        user_text_profiles[user] = np.asarray(text_matrix_normalized[liked_indices].mean(axis=0))

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

gamma = 0.25 #same gamma weight from class examples

test_users = test_df['user_id'].unique()
all_businesses = restaurants_df['business_id'].tolist()
predictions = {}

for user in test_users:
    if user in user_meta_profiles and user in user_text_profiles:
        #CALCULATE META DATA SIMS
        sim_meta = cosine_similarity(user_meta_profiles[user].reshape(1, -1), feature_matrix_normalized).flatten()

        #CALCULATE TXT SIMS
        sim_text = cosine_similarity(user_text_profiles[user].reshape(1, -1), text_matrix_normalized).flatten()

        #WEIGHT THEM
        final_sim_scores = (gamma * sim_text) + ((1 - gamma) * sim_meta)

        #Rank top 30
        top_indices = final_sim_scores.argsort()[-30:][::-1]
        predictions[user] = [all_businesses[i] for i in top_indices]
    else:
        predictions[user] = []

In [14]:
import pandas as pd

metrics = evaluate_model(predictions, test_df)

w_text = gamma
w_meta = round(1 - gamma, 2)

results_df = pd.DataFrame([metrics]).round(4)
results_df.index = [f'Content-Based B (Meta={w_meta}, Text={w_text})']
display(results_df)

,Hit@10,Hit@20,Hit@30,NDCG@10,NDCG@20,NDCG@30
"Content-Based B (Meta=0.75, Text=0.25)",0.0414,0.076,0.1083,0.0169,0.0255,0.0323
